# Adaptible self-repair cycles on a Colab GPU

Runtime → Change runtime type → **T4 GPU** (free tier). Then **Runtime → Run all**.

Every cycle checkpoints to `MyDrive/adaptible/cycles_t4/` (LoRA weights, history, status).
If the session dies, just **Run all** again: the script resumes from the last completed cycle.
`status.json` in that folder is what gets polled from outside to see whether the run is alive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/adaptible/cycles_t4'
import os; os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers peft accelerate
!rm -rf /content/adaptible && git clone -q https://github.com/ible-ai/adaptible /content/adaptible
!cd /content/adaptible && git log --oneline -1

In [ ]:
# Runs (or resumes) the loop; the log also goes to Drive so it survives the session.
!cd /content/adaptible && python -u scripts/colab/cycles_torch.py --out "$OUT" --cycles 40 2>&1 | tee -a "$OUT/run.log" | grep --line-buffered -E "^(model=|RESUME|BASE|CAND|NOCAND|CYCLE|SUMMARY)|Traceback|Error"